<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/04%20-%20Bayesian%20Optimization/solutions/04_bo_solutions.ipynb)

# 第四部分参考答案：贝叶斯优化

包含 `04a_gaussian_processes.ipynb` 和 `04b_bayesian_optimization.ipynb` 全部 6 道 🏋️ 练习的参考代码与讨论。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "04 - Bayesian Optimization/solutions"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from sklearn.decomposition import PCA
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110

def find_file(name):
    for path in (f"../data/{name}", f"data/{name}", name):
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"找不到 {name}")

## 练习 04a-1：比较不同核函数的光滑度

In [ ]:
X_train = np.array([1.0, 2.2, 3.5, 7.2, 8.5, 9.3]).reshape(-1, 1)
def true_mobility(x):
    return np.sin(0.9 * x) + 0.3 * np.cos(2.1 * x)

rng = np.random.RandomState(7)
y_noisy = true_mobility(X_train).ravel() + rng.normal(0, 0.15, size=len(X_train))
X_grid = np.linspace(0, 10, 100).reshape(-1, 1)

kernels = {
    "Matern (nu=0.5, rough)": C(1.0) * Matern(1.0, nu=0.5) + WhiteKernel(0.1),
    "Matern (nu=1.5, moderate)": C(1.0) * Matern(1.0, nu=1.5) + WhiteKernel(0.1),
    "RBF (infinitely smooth)": C(1.0) * RBF(1.0) + WhiteKernel(0.1),
}

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), sharey=True)
for ax, (name, kern) in zip(axes, kernels.items()):
    gp = GaussianProcessRegressor(kernel=kern, n_restarts_optimizer=10, random_state=0)
    gp.fit(X_train, y_noisy)
    mu, std = gp.predict(X_grid, return_std=True)
    ax.plot(X_grid, true_mobility(X_grid), "k--", alpha=0.5, label="True curve")
    ax.plot(X_grid, mu, "b-", lw=2, label="Mean")
    ax.fill_between(X_grid.ravel(), mu - 1.96 * std, mu + 1.96 * std, color="blue", alpha=0.15)
    ax.scatter(X_train, y_noisy, c="red", s=50, zorder=5)
    ax.set_title(name, fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

📘 `nu=0.5` 会画出折线般尖锐的均值，且在偏离观测点后置信区间迅速张开；`nu=1.5` 和 `RBF` 更光滑，更符合多数连续热力学/器件退火曲线的物理先验。

## 练习 04a-2：实验点数量如何改变不确定性？

In [ ]:
def rbf_kernel(X1, X2, length_scale=1.2, sigma_f=1.0):
    sq_dist = np.sum(X1**2, axis=1, keepdims=True) + np.sum(X2**2, axis=1) - 2 * X1 @ X2.T
    return (sigma_f ** 2) * np.exp(-0.5 * sq_dist / (length_scale ** 2))

def gp_posterior(X_tr, y_tr, X_te, length_scale=1.2, sigma_f=1.0, noise=1e-4):
    K = rbf_kernel(X_tr, X_tr, length_scale, sigma_f) + noise * np.eye(len(X_tr))
    K_s = rbf_kernel(X_tr, X_te, length_scale, sigma_f)
    K_ss = rbf_kernel(X_te, X_te, length_scale, sigma_f) + 1e-8 * np.eye(len(X_te))
    K_inv = np.linalg.inv(K)
    mu_s = K_s.T @ K_inv @ y_tr
    cov_s = K_ss - K_s.T @ K_inv @ K_s
    return mu_s, np.sqrt(np.maximum(np.diag(cov_s), 0))

X_tr_6 = X_train
X_tr_12 = np.sort(np.vstack([X_train, np.array([[0.3], [4.2], [5.0], [5.8], [6.5], [9.8]])]), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, X_tr, label in zip(axes, [X_tr_6, X_tr_12], ["n = 6", "n = 12"]):
    y_tr = true_mobility(X_tr).ravel()
    mu_s, std_s = gp_posterior(X_tr, y_tr, X_grid)
    print(f"{label} 平均不确定性 mean(std) = {std_s.mean():.4f}")
    ax.plot(X_grid, true_mobility(X_grid), "k--", alpha=0.5)
    ax.plot(X_grid, mu_s, "b-", lw=2)
    ax.fill_between(X_grid.ravel(), mu_s - 1.96 * std_s, mu_s + 1.96 * std_s, color="blue", alpha=0.18)
    ax.scatter(X_tr, y_tr, c="red", s=45, zorder=5)
    ax.set_title(f"{label} (mean std = {std_s.mean():.3f})")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 练习 04a-3：预测有机半导体分子的溶解度及不确定性

In [ ]:
df = pd.read_csv(find_file("solubility.csv"))
sub = df.sample(700, random_state=42).reset_index(drop=True)
fps, ys = [], []
for smi, logs in zip(sub["SMILES"], sub["logS"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fps.append(np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048), dtype=float))
        ys.append(logs)
    if len(fps) == 400:
        break
X_tr_fp = np.array(fps)
y_tr_fp = np.array(ys)

def tanimoto_kernel(A, B):
    dot = A @ B.T
    na = np.sum(A**2, axis=1, keepdims=True)
    nb = np.sum(B**2, axis=1, keepdims=True).T
    denom = na + nb - dot
    return np.where(denom > 0, dot / denom, 0.0)

K_tr = tanimoto_kernel(X_tr_fp, X_tr_fp) + 0.15 * np.eye(len(X_tr_fp))
K_inv = np.linalg.inv(K_tr)
y_mean = y_tr_fp.mean()

osc_mols = {
    "Benzene": "c1ccccc1",
    "Thiophene": "c1ccsc1",
    "Anthracene": "c1ccc2cc3ccccc3cc2c1",
    "Tetracene": "c1ccc2cc3cc4ccccc4cc3cc2c1",
    "C8-BTBT": "CCCCCCCCc1ccc2c(c1)sc1c2sc2cc(CCCCCCCC)ccc21",
}
X_osc = np.array([np.array(AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, 2048), dtype=float)
                  for s in osc_mols.values()])
K_s = tanimoto_kernel(X_tr_fp, X_osc)
mu_osc = y_mean + K_s.T @ K_inv @ (y_tr_fp - y_mean)
std_osc = np.sqrt(np.maximum(1.0 - np.sum((K_s.T @ K_inv) * K_s.T, axis=1), 0))
max_sim = K_s.max(axis=0)

pd.DataFrame({
    "Molecule": list(osc_mols.keys()),
    "Pred logS": np.round(mu_osc, 2),
    "Pred std": np.round(std_osc, 3),
    "Max Tanimoto Sim in Train": np.round(max_sim, 3)
})

📘 可以看到，在训练集里存在大量相似分子（`Max Tanimoto Sim` 较高）的苯或蒽，不确定性更低；而长链稠环 `C8-BTBT` 离 400 个小分子样本较远，不确定性 $\sigma$ 接近最高值（约 0.99），几乎退回先验均值附近。

## 练习 04b-1：调节 UCB 的探索系数 $\kappa$

In [ ]:
def hidden_process(x):
    return 0.6 * np.exp(-0.5 * ((x - 2.3) / 0.8)**2) + 1.0 * np.exp(-0.5 * ((x - 7.8) / 0.9)**2)

X_grid = np.linspace(0, 10, 300).reshape(-1, 1)
X_init = np.array([[1.2], [2.2], [3.6]])
y_init = hidden_process(X_init).ravel()

for kappa in [0.1, 2.0, 10.0]:
    X_obs, y_obs = X_init.copy(), y_init.copy()
    for step in range(1, 7):
        gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-5, random_state=0)
        gp.fit(X_obs, y_obs)
        mu, std = gp.predict(X_grid, return_std=True)
        next_x = X_grid[np.argmax(mu + kappa * std)]
        next_y = hidden_process(next_x).item()
        X_obs = np.vstack([X_obs, next_x.reshape(1, 1)])
        y_obs = np.append(y_obs, next_y)
    print(f"kappa={kappa:>4} | 6步后找到的最大值 = {y_obs.max():.4f} (全局最优=1.0002), 采样点 = {np.round(X_obs[3:].ravel(), 2)}")

📘 `kappa=0.1` 几乎纯粹利用，一直困在左边 `x ≈ 2.3` 的小山头（最大值只有 0.60）；`kappa=2.0` 与 `kappa=10.0` 则成功跳出局部最优找到了右边 `x ≈ 7.8` 的全局主峰！

## 练习 04b-2：二维材料工艺参数寻优

In [ ]:
def process_2d(X):
    # X: (n, 2), 第一列温度 T，第二列浓度 c
    t, c = X[:, 0], X[:, 1]
    peak1 = 0.65 * np.exp(-0.3 * ((t - 2.5)**2 + (c - 3.0)**2))
    peak2 = 1.00 * np.exp(-0.35 * ((t - 7.5)**2 + (c - 7.2)**2))
    return peak1 + peak2

def acq_ei(mu, std, y_best, xi=0.01):
    with np.errstate(divide="ignore", invalid="ignore"):
        imp = mu - y_best - xi
        Z = np.where(std > 1e-9, imp / std, 0.0)
        return np.where(std > 1e-9, imp * norm.cdf(Z) + std * norm.pdf(Z), 0.0)

g = np.linspace(0, 10, 25)
TT, CC = np.meshgrid(g, g)
X_cand = np.column_stack([TT.ravel(), CC.ravel()])
y_true_all = process_2d(X_cand)

rng = np.random.RandomState(42)
init_idx = rng.choice(len(X_cand), size=3, replace=False)
obs_idx = list(init_idx)
for _ in range(15):
    gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.5), alpha=1e-5, random_state=0)
    gp.fit(X_cand[obs_idx], y_true_all[obs_idx])
    unseen = [i for i in range(len(X_cand)) if i not in obs_idx]
    mu, std = gp.predict(X_cand[unseen], return_std=True)
    ei = acq_ei(mu, std, y_true_all[obs_idx].max())
    obs_idx.append(unseen[np.argmax(ei)])

rand_idx = rng.choice(len(X_cand), size=18, replace=False)
print(f"18 次实验总计: BO 最大值 = {y_true_all[obs_idx].max():.4f} vs 随机盲试最大值 = {y_true_all[rand_idx].max():.4f}")

plt.figure(figsize=(6, 5))
plt.contourf(TT, CC, y_true_all.reshape(25, 25), levels=20, cmap="viridis")
plt.colorbar(label="Performance")
plt.scatter(X_cand[obs_idx[:3], 0], X_cand[obs_idx[:3], 1], c="white", edgecolors="black", s=60, label="Init (n=3)")
plt.scatter(X_cand[obs_idx[3:], 0], X_cand[obs_idx[3:], 1], c="red", edgecolors="white", s=70, marker="*", label="BO Selected (n=15)")
plt.xlabel("Temperature T")
plt.ylabel("Concentration c")
plt.title("2D Bayesian Optimization Trajectory")
plt.legend()
plt.show()

## 练习 04b-3：在 Buchwald-Hartwig 上对比分子指纹+PCA 与 One-Hot

In [ ]:
bh = pd.read_csv(find_file("bh-reactions.csv"))
y_yield = bh["yield"].values

# 对 4 个组分分别算唯一 SMILES 的 256-bit 指纹再查表拼接，速度极快
def col_to_fp(series, n_bits=256):
    uniq = series.unique()
    cache = {s: np.array(AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, n_bits), dtype=float) for s in uniq}
    return np.vstack([cache[s] for s in series])

X_fp_raw = np.hstack([col_to_fp(bh[c]) for c in ["ligand", "additive", "base", "aryl halide"]])
X_fp_pca = PCA(n_components=30, random_state=0).fit_transform(X_fp_raw)
X_oh = pd.get_dummies(bh[["ligand", "additive", "base", "aryl halide"]], dtype=float).values

def run_bo(X, y, seeds=(0, 1, 2), n_init=5, n_steps=20):
    all_curves = []
    for s in seeds:
        rng = np.random.RandomState(s)
        obs = list(rng.choice(len(y), size=n_init, replace=False))
        hist = [y[obs].max()]
        for _ in range(n_steps):
            gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(2.0) + WhiteKernel(1.0), normalize_y=True, random_state=s)
            gp.fit(X[obs], y[obs])
            unseen = np.array([i for i in range(len(y)) if i not in obs])
            mu, std = gp.predict(X[unseen], return_std=True)
            scores = acq_ei(mu, std, y[obs].max(), xi=0.5)
            obs.append(unseen[np.argmax(scores)])
            hist.append(y[obs].max())
        all_curves.append(hist)
    return np.array(all_curves)

c_oh = run_bo(X_oh, y_yield)
c_fp = run_bo(X_fp_pca, y_yield)
print(f"25 次实验后平均最高产率: One-Hot = {c_oh[:,-1].mean():.1f}% vs Morgan-PCA = {c_fp[:,-1].mean():.1f}%")